# UNIVERSIDAD EXTERNADO DE COLOMBIA · CIENCIA DE DATOS · 2026

## Modelado de una sola etapa

**Predicción de la Estancia Hospitalaria con Aprendizaje Automático — SPARCS CY2021**

Autores: Daniel Vallejo Vecil · Juliana M.

---

Este cuaderno entrena y compara los modelos de una sola etapa, que estiman la duración de la estancia
directamente sobre la distribución completa del desenlace. Cumple dos funciones en el proyecto:
constituye la referencia frente a la cual se evalúa el esquema en dos fases del cuaderno 08, y
permite establecer cuál es la función de pérdida más adecuada a la naturaleza de la variable
objetivo.

Todos los modelos se entrenan sobre la **variable objetivo en días**, sin transformación, y todas las
métricas se reportan en esa misma escala.

## 1. Fundamento

La duración de la estancia es una variable de conteo, estrictamente positiva y fuertemente asimétrica
a la derecha: asimetría de 5,82, curtosis de 52,96, media de 5,75 días frente a una mediana de 3.
Bajo error cuadrático, los episodios de la cola dominan la función de pérdida y el modelo sacrifica
ajuste en el grueso de la distribución para reducir el error en unos pocos casos extremos.

La estrategia adoptada para corregirlo consiste en sustituir el error cuadrático por una función de
pérdida acorde a la naturaleza de conteo del desenlace. Las pérdidas de Poisson, Gamma y Tweedie
incorporan un enlace logarítmico de forma nativa: el modelo recibe y devuelve la estancia en días,
pero modela internamente el logaritmo de la estancia esperada. Por esa razón sus predicciones no
requieren ninguna corrección de retransformación.

La pérdida de Tweedie resulta particularmente apropiada porque su parámetro de varianza permite
transitar de forma continua entre el comportamiento de Poisson (parámetro igual a 1) y el de Gamma
(parámetro igual a 2), de modo que la especificación se ajusta empíricamente a la forma de la
distribución observada en lugar de imponerse a priori.

Se incluye además, como referencia, la pérdida de error absoluto, con el fin de cuantificar qué se
alcanza optimizando directamente el MAE y a costa de qué.

### Consecuencia para el análisis de interpretabilidad

El enlace logarítmico de estas pérdidas tiene una implicación que conviene anticipar: las
atribuciones SHAP de estos modelos se expresan en la escala del logaritmo y se interpretan como
factores que multiplican la estancia esperada, no como días sumados o restados. El cuaderno de
interpretabilidad lo trata explícitamente.

## 2. Configuración y carga

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib, time

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().parent
GOLD_PATH = PROJECT_ROOT / "data" / "3_gold"
MODELOS_PATH = PROJECT_ROOT / "modelos"
TABLAS_PATH = PROJECT_ROOT / "tablas"
MODELOS_PATH.mkdir(parents=True, exist_ok=True)
TABLAS_PATH.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NUM = "05"


def guardar_tabla(tabla, nombre):
    if isinstance(tabla, pd.Series):
        tabla = tabla.to_frame()
    archivo = TABLAS_PATH / f"{NOTEBOOK_NUM}_{nombre}.csv"
    tabla.to_csv(archivo, encoding="utf-8-sig")
    print(f"Tabla guardada: {archivo.resolve()}")


# El split viene fijado desde 03_FeatureEngineering. NO se vuelve a particionar.
X_train = pd.read_parquet(GOLD_PATH / "X_train_sparcs.parquet")
X_test = pd.read_parquet(GOLD_PATH / "X_test_sparcs.parquet")
y_train = pd.read_parquet(GOLD_PATH / "y_train_sparcs.parquet")["Length of Stay"]
y_test = pd.read_parquet(GOLD_PATH / "y_test_sparcs.parquet")["Length of Stay"]

print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")

X_train: (1672271, 189)   y_train: (1672271,)
X_test:  (418068, 189)   y_test:  (418068,)


## 3. Caracterización del desenlace

Los estadísticos que motivan la elección de la función de pérdida.

In [2]:
resumen = pd.DataFrame({
    "valor": [y_train.mean(), y_train.median(), y_train.std(), y_train.min(), y_train.max(),
              y_train.skew(), y_train.kurtosis(),
              (y_train == 1).mean() * 100, (y_train > 7).mean() * 100,
              (y_train >= 120).mean() * 100]
}, index=["media", "mediana", "desv_std", "mínimo", "máximo", "asimetría", "curtosis",
          "% estancias de 1 día", "% superiores a 7 días", "% en la categoría censurada"])

display(resumen.round(3))
guardar_tabla(resumen, "CaracterizacionDesenlace")

,valor
media,5.744
mediana,3.000
desv_std,8.378
mínimo,1.000
máximo,120.000
asimetría,5.825
curtosis,53.170
% estancias de 1 día,17.455
% superiores a 7 días,20.124
% en la categoría censurada,0.073


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\05_CaracterizacionDesenlace.csv


## 4. Función de evaluación

Todas las métricas se calculan en días sobre el conjunto de prueba completo. Las predicciones
inferiores a un día se ajustan a ese valor, que es el mínimo observable de la variable.

In [3]:
resultados = []


def evaluar(nombre, y_real, y_pred, nota=""):
    """Calcula MAE, RMSE y R2 en escala de días y acumula el resultado."""
    y_pred = np.clip(y_pred, 1, None)
    fila = {
        "modelo": nombre,
        "MAE": mean_absolute_error(y_real, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_real, y_pred))),
        "R2": r2_score(y_real, y_pred),
        "pred_media": float(np.mean(y_pred)),
        "nota": nota,
    }
    resultados.append(fila)
    print(f"{nombre:34s}  MAE={fila['MAE']:.3f}  RMSE={fila['RMSE']:.3f}  R2={fila['R2']:.4f}")
    return fila

## 5. Comparación entre familias de algoritmos

Las tres familias con error cuadrático, que es la especificación de referencia. Los hiperparámetros
se mantienen fijos en todo el cuaderno, de modo que las diferencias observadas más adelante sean
atribuibles a la función de pérdida y no a la capacidad del modelo.

In [4]:
PARAMS_XGB = dict(
    n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, tree_method="hist", random_state=42, n_jobs=-1,
)
PARAMS_RF = dict(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)

modelos_base = {
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(**PARAMS_RF),
    "XGBoost (error cuadrático)": XGBRegressor(objective="reg:squarederror", **PARAMS_XGB),
}

modelos_entrenados = {}

for nombre, modelo in modelos_base.items():
    t0 = time.time()
    print(f"Entrenando: {nombre} ...")
    modelo.fit(X_train, y_train)
    evaluar(nombre, y_test, modelo.predict(X_test), nota="error cuadrático")
    modelos_entrenados[nombre] = modelo
    slug = nombre.split(" ")[0].lower()
    joblib.dump(modelo, MODELOS_PATH / f"05_{slug}.joblib")
    print(f"   ({time.time()-t0:.0f}s)")

Entrenando: Ridge ...
Ridge                               MAE=3.271  RMSE=6.516  R2=0.3840
   (11s)
Entrenando: Random Forest ...
Random Forest                       MAE=2.947  RMSE=6.183  R2=0.4454
   (8401s)
Entrenando: XGBoost (error cuadrático) ...
XGBoost (error cuadrático)          MAE=2.933  RMSE=6.086  R2=0.4627
   (132s)


## 6. Funciones de pérdida adaptadas a datos de conteo

Sobre XGBoost, manteniendo idénticos el resto de hiperparámetros, se sustituye el error cuadrático
por pérdidas con enlace logarítmico. El parámetro de varianza de Tweedie se explora en una rejilla
corta entre los extremos de Poisson y Gamma.

In [5]:
objetivos = {
    "XGBoost Poisson": dict(objective="count:poisson"),
    "XGBoost Gamma": dict(objective="reg:gamma"),
    "XGBoost Tweedie p=1.2": dict(objective="reg:tweedie", tweedie_variance_power=1.2),
    "XGBoost Tweedie p=1.5": dict(objective="reg:tweedie", tweedie_variance_power=1.5),
    "XGBoost Tweedie p=1.8": dict(objective="reg:tweedie", tweedie_variance_power=1.8),
    "XGBoost MAE": dict(objective="reg:absoluteerror"),
}

for nombre, extra in objetivos.items():
    t0 = time.time()
    print(f"Entrenando: {nombre} ...")
    modelo = XGBRegressor(**{**PARAMS_XGB, **extra})
    modelo.fit(X_train, y_train)
    evaluar(nombre, y_test, modelo.predict(X_test), nota=str(extra))
    modelos_entrenados[nombre] = modelo
    joblib.dump(modelo, MODELOS_PATH / f"05_{nombre.replace(' ', '_').replace('=', '').lower()}.joblib")
    print(f"   ({time.time()-t0:.0f}s)")

Entrenando: XGBoost Poisson ...
XGBoost Poisson                     MAE=2.946  RMSE=6.112  R2=0.4579
   (155s)
Entrenando: XGBoost Gamma ...
XGBoost Gamma                       MAE=2.885  RMSE=6.098  R2=0.4605
   (143s)
Entrenando: XGBoost Tweedie p=1.2 ...
XGBoost Tweedie p=1.2               MAE=2.906  RMSE=6.072  R2=0.4651
   (143s)
Entrenando: XGBoost Tweedie p=1.5 ...
XGBoost Tweedie p=1.5               MAE=2.896  RMSE=6.074  R2=0.4646
   (143s)
Entrenando: XGBoost Tweedie p=1.8 ...
XGBoost Tweedie p=1.8               MAE=2.889  RMSE=6.091  R2=0.4617
   (142s)
Entrenando: XGBoost MAE ...
XGBoost MAE                         MAE=2.782  RMSE=6.439  R2=0.3984
   (413s)


## 7. Comparación consolidada

In [6]:
comparacion = pd.DataFrame(resultados).sort_values("R2", ascending=False).reset_index(drop=True)
display(comparacion.round(4))
guardar_tabla(comparacion, "ComparacionUnaEtapa")

mejor_r2 = comparacion.iloc[0]
mejor_mae = comparacion.sort_values("MAE").iloc[0]
print()
print(f"Mayor R2:   {mejor_r2['modelo']}  (R2={mejor_r2['R2']:.4f}, MAE={mejor_r2['MAE']:.3f})")
print(f"Menor MAE:  {mejor_mae['modelo']}  (MAE={mejor_mae['MAE']:.3f}, R2={mejor_mae['R2']:.4f})")
print()
print(f"Estancia media real en el conjunto de prueba: {y_test.mean():.2f} días")
print("Comparar con la columna pred_media: los modelos que minimizan el MAE lo consiguen")
print("prediciendo por debajo de la media real, es decir, subestimando la cola.")

,modelo,MAE,RMSE,R2,pred_media,nota
0,XGBoost Tweedie p=1.2,2.9056,6.0719,0.4651,5.6850,"{'objective': 'reg:tweedie', 'tweedie_variance..."
1,XGBoost Tweedie p=1.5,2.8959,6.0744,0.4646,5.6532,"{'objective': 'reg:tweedie', 'tweedie_variance..."
2,XGBoost (error cuadrático),2.9327,6.0856,0.4627,5.7168,error cuadrático
3,XGBoost Tweedie p=1.8,2.8893,6.0911,0.4617,5.6245,"{'objective': 'reg:tweedie', 'tweedie_variance..."
4,XGBoost Gamma,2.8847,6.0981,0.4605,5.6046,{'objective': 'reg:gamma'}
5,XGBoost Poisson,2.9459,6.1123,0.4579,5.7120,{'objective': 'count:poisson'}
6,Random Forest,2.9467,6.1829,0.4454,5.7133,error cuadrático
7,XGBoost MAE,2.7824,6.4394,0.3984,4.5915,{'objective': 'reg:absoluteerror'}
8,Ridge,3.2714,6.5160,0.3840,5.7609,error cuadrático


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\05_ComparacionUnaEtapa.csv

Mayor R2:   XGBoost Tweedie p=1.2  (R2=0.4651, MAE=2.906)
Menor MAE:  XGBoost MAE  (MAE=2.782, R2=0.3984)

Estancia media real en el conjunto de prueba: 5.72 días
Comparar con la columna pred_media: los modelos que minimizan el MAE lo consiguen
prediciendo por debajo de la media real, es decir, subestimando la cola.


Es esperable que el modelo con menor MAE y el de mayor $R^2$ **no coincidan**. El error absoluto
premia acertar en el grueso de la distribución —las estancias cortas, que son la mayoría—, mientras
que el $R^2$ penaliza cuadráticamente los errores de la cola. La columna `pred_media` permite ver el
mecanismo: los modelos con MAE bajo predicen sistemáticamente por debajo de la estancia media real.

Por esa razón el criterio de selección del proyecto adopta el $R^2$ y el RMSE como métricas
principales, y el MAE como métrica complementaria: en un problema de gestión de camas, subestimar
una estancia prolongada tiene un costo operativo mucho mayor que sobreestimar una breve.

## 8. Desempeño por tramo de estancia

Una métrica global oculta dónde acierta y dónde falla cada modelo. Esta descomposición es el
resultado más informativo del cuaderno y se retoma en la discusión.

In [7]:
TRAMOS = [(1, 2, "1-2 días"), (3, 6, "3-6 días"), (7, 13, "7-13 días"),
          (14, 29, "14-29 días"), (30, 119, "30-119 días"), (120, 120, "120+ (censurado)")]

MODELOS_A_COMPARAR = ["XGBoost (error cuadrático)", "XGBoost Tweedie p=1.5", "XGBoost MAE"]

filas = []
for nombre in MODELOS_A_COMPARAR:
    modelo = modelos_entrenados.get(nombre)
    if modelo is None:
        continue
    pred = np.clip(modelo.predict(X_test), 1, None)

    for lo, hi, etiqueta in TRAMOS:
        m = ((y_test >= lo) & (y_test <= hi)).to_numpy()
        if m.sum() == 0:
            continue
        filas.append({
            "modelo": nombre, "tramo": etiqueta, "n": int(m.sum()),
            "pct_test": m.mean() * 100,
            "real_medio": float(y_test.to_numpy()[m].mean()),
            "pred_media": float(pred[m].mean()),
            "MAE": mean_absolute_error(y_test.to_numpy()[m], pred[m]),
        })

por_tramo = pd.DataFrame(filas)
display(por_tramo.round(2))
guardar_tabla(por_tramo, "DesempenoPorTramo")

,modelo,tramo,n,pct_test,real_medio,pred_media,MAE
0,XGBoost (error cuadrático),1-2 días,165259,39.53,1.56,3.16,1.66
1,XGBoost (error cuadrático),3-6 días,152033,36.37,4.06,5.01,1.90
2,XGBoost (error cuadrático),7-13 días,64491,15.43,9.16,8.52,3.44
3,XGBoost (error cuadrático),14-29 días,28338,6.78,18.94,12.96,7.87
4,XGBoost (error cuadrático),30-119 días,7658,1.83,46.35,23.07,25.48
5,XGBoost (error cuadrático),120+ (censurado),289,0.07,120.00,41.52,78.58
6,XGBoost Tweedie p=1.5,1-2 días,165259,39.53,1.56,3.11,1.60
7,XGBoost Tweedie p=1.5,3-6 días,152033,36.37,4.06,4.96,1.86
8,XGBoost Tweedie p=1.5,7-13 días,64491,15.43,9.16,8.46,3.44
9,XGBoost Tweedie p=1.5,14-29 días,28338,6.78,18.94,12.87,7.89


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\05_DesempenoPorTramo.csv


## 9. Modelo final y preparación del análisis de interpretabilidad

Se persiste el modelo seleccionado junto con una muestra del conjunto de prueba, que consume el
cuaderno de interpretabilidad. La elección de la variante se documenta en el Capítulo 4; aquí solo
se materializa.

In [8]:
MODELO_FINAL = "XGBoost Tweedie p=1.5"

modelo_final = modelos_entrenados[MODELO_FINAL]
joblib.dump(modelo_final, MODELOS_PATH / "05_modelo_final.joblib")

# Muestra del conjunto de prueba para el análisis SHAP
muestra = X_test.sample(n=min(50_000, len(X_test)), random_state=42)
muestra.to_parquet(MODELOS_PATH / "05_muestra_shap.parquet")
y_test.loc[muestra.index].to_frame().to_parquet(MODELOS_PATH / "05_muestra_shap_y.parquet")

fila = [r for r in resultados if r["modelo"] == MODELO_FINAL][0]
print(f"Modelo final: {MODELO_FINAL}")
print(f"  MAE  = {fila['MAE']:.3f} días")
print(f"  RMSE = {fila['RMSE']:.3f} días")
print(f"  R2   = {fila['R2']:.4f}")
print()
print("Modelo y muestra guardados en /modelos.")

Modelo final: XGBoost Tweedie p=1.5
  MAE  = 2.896 días
  RMSE = 6.074 días
  R2   = 0.4646

Modelo y muestra guardados en /modelos.


## 10. Conclusiones del cuaderno

Elementos a documentar en el Capítulo 4 a partir de las tablas generadas:

1. **La comparación entre familias con error cuadrático**, que cuantifica la ganancia de los métodos
   no lineales sobre la regresión regularizada.
2. **El efecto de la función de pérdida**: si las pérdidas con enlace logarítmico superan al error
   cuadrático sobre el mismo algoritmo, y qué valor del parámetro de varianza de Tweedie resulta
   más apropiado para esta distribución.
3. **La tensión entre el MAE y el $R^2$**, evidenciada por la pérdida de error absoluto: alcanza el
   menor MAE de la comparación a costa de un $R^2$ sustancialmente inferior, y lo hace prediciendo
   por debajo de la estancia media real.
4. **El patrón por tramos**: sobreestimación en los tramos cortos y subestimación creciente en los
   largos, con el tramo censurado como el de mayor error. Ese patrón es el resultado sustantivo y
   coincide con lo reportado por Xu et al. (2022).

**Tablas exportadas:** `05_CaracterizacionDesenlace`, `05_ComparacionUnaEtapa`,
`05_DesempenoPorTramo`.